In [1]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.append("..")
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import PoissonRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.evaluation import load_train, poisson_deviance, cross_validate, PROCESSED, N_FOLDS
from src.features import add_features, NUM, FLAGS, CAT

train = load_train()
y = train["ClaimNb"].to_numpy(float)
masks = [(train["Fold"] == f).to_numpy() for f in range(N_FOLDS)]

def fold_scores(pred):
    return np.array([poisson_deviance(y[m], pred[m]) for m in masks])

# References from the saved out-of-fold predictions (no refit needed)
s_base = fold_scores(pd.read_csv(PROCESSED / "oof_glm.csv")["pred_claims"].to_numpy())     # GLM, alpha 1e-4
s_gbm = fold_scores(pd.read_csv(PROCESSED / "oof_gbm.csv")["pred_claims"].to_numpy())
AVG_GAP = (s_base - s_gbm).mean()
print(f"GLM {s_base.mean():.4f} | boosting {s_gbm.mean():.4f} | gap to close {AVG_GAP:.4f}")      # about 0.3121 | 0.3038 | 0.0083

# Candidate interactions: each is 1 only for new vehicles (age 0) in a given subgroup
INTERACTIONS = {
    "age0_B12":        lambda d, X: ((d["VehAge"] == 0) & (d["VehBrand"] == "B12")).astype(int),
    "age0_Regular":    lambda d, X: ((d["VehAge"] == 0) & (d["VehGas"] == "Regular")).astype(int),
    "age0_off_ladder": lambda d, X: ((d["VehAge"] == 0) & (X["bm_off_ladder"] == 1)).astype(int),
    "age0_bm100":      lambda d, X: ((d["VehAge"] == 0) & (X["bm_is_100"] == 1)).astype(int),
}

def features_x(d, extras):
    X = add_features(d)                                  # the shared row-wise features
    for name in extras:
        X[name] = INTERACTIONS[name](d, X)
    return X

def glm_x_fit_predict(tr, va, extras, alpha=1e-4):
    model = Pipeline([
        ("pre", ColumnTransformer([
            ("num",  StandardScaler(), NUM),
            ("flag", "passthrough", FLAGS + extras),      # interactions are 0/1 flags: passed through unscaled
            ("cat",  OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=0.0037), CAT),
        ])),
        ("glm", PoissonRegressor(alpha=alpha, max_iter=1000)),
    ])
    model.fit(features_x(tr, extras), tr["ClaimNb"] / tr["Exposure"], glm__sample_weight=tr["Exposure"])
    return model.predict(features_x(va, extras)) * va["Exposure"].to_numpy()

VARIANTS = {
    "+ age0_B12": ["age0_B12"],
    "+ age0_Regular": ["age0_Regular"],
    "+ B12 + Regular": ["age0_B12", "age0_Regular"],
    "+ all four": ["age0_B12", "age0_Regular", "age0_off_ladder", "age0_bm100"],
}
rows, oofs = [], {}
for name, extras in VARIANTS.items():
    oofs[name], s = cross_validate(train, lambda tr, va, ex=extras: glm_x_fit_predict(tr, va, ex), check=False)
    rows.append({"variant": name, "mean_deviance": s.mean(), "gain_vs_glm": (s_base - s).mean(),
                 "folds_better": int((s < s_base).sum()), "worst_fold_gain": (s_base - s).min(),
                 "gap_closed": (s_base - s).mean() / AVG_GAP})
    print(name, "done")
print(pd.DataFrame(rows).round(4).to_string(index=False))

GLM 0.3121 | boosting 0.3038 | gap to close 0.0083
+ age0_B12 done
+ age0_Regular done
+ B12 + Regular done
+ all four done
        variant  mean_deviance  gain_vs_glm  folds_better  worst_fold_gain  gap_closed
     + age0_B12         0.3098       0.0022             5           0.0021      0.2694
 + age0_Regular         0.3109       0.0012             5           0.0010      0.1428
+ B12 + Regular         0.3090       0.0031             5           0.0028      0.3696
     + all four         0.3083       0.0037             5           0.0033      0.4512


In [2]:
from src.evaluation import EXPOSURE_EDGES
from src.features import LADDER, DRIVAGE_EDGES, DRIVAGE_LABELS, VEHAGE_EDGES, VEHAGE_LABELS

best = "+ all four"
mu_x = oofs[best]
pd.DataFrame({"IDpol": train["IDpol"], "pred_claims": mu_x}).to_csv(PROCESSED / "oof_glm_x.csv", index=False)
mu_gbm = pd.read_csv(PROCESSED / "oof_gbm.csv")["pred_claims"].to_numpy()

def unit_deviance(y, mu):
    """Per-policy deviance contribution; its mean is the poisson_deviance used everywhere."""
    term = np.zeros_like(y); pos = y > 0
    term[pos] = y[pos] * np.log(y[pos] / mu[pos])
    return 2 * (term - (y - mu))

diff_x = unit_deviance(y, mu_x) - unit_deviance(y, mu_gbm)       # positive = boosting better on that policy
AVG = diff_x.mean()
print("remaining gap per policy:", round(AVG, 5))              # expect about 0.0046

def report(mask, key):
    """Remaining-gap breakdown for the policies in `mask`, by the levels of `key`."""
    g = pd.DataFrame({"key": key[mask].to_numpy(), "d": diff_x[mask], "y": y[mask],
                      "glm": mu_x[mask], "gbm": mu_gbm[mask]}).groupby("key", observed=True).agg(
        policies=("y", "size"), claims=("y", "sum"), glm=("glm", "sum"), gbm=("gbm", "sum"), d=("d", "sum"))
    g["share_of_policies"] = g["policies"] / g["policies"].sum()
    g["share_of_gap"] = g["d"] / g["d"].sum()
    g["gap_vs_avg"] = (g["d"] / g["policies"]) / AVG              # 1 = fair share; far above 1 = where boosting still wins
    g["actual/GLMx"] = g["claims"] / g["glm"]                      # calibration of the improved GLM inside the segment
    g["actual/GBM"] = g["claims"] / g["gbm"]
    return g[["policies", "share_of_policies", "share_of_gap", "gap_vs_avg", "actual/GLMx", "actual/GBM"]].round(3)

bm = train["BonusMalus"]
keys = {
    "VehAge band": pd.cut(train["VehAge"], VEHAGE_EDGES, labels=VEHAGE_LABELS),
    "DrivAge band": pd.cut(train["DrivAge"], DRIVAGE_EDGES, labels=DRIVAGE_LABELS),
    "BonusMalus class": pd.Series(np.select([bm == 50, bm == 100, ~bm.isin(LADDER)], ["50", "100", "off ladder"],
                                            default="ladder 51-95"), index=train.index),
    "Brand": pd.Series(np.where(train["VehBrand"] == "B12", "B12", "other"), index=train.index),
    "VehGas": train["VehGas"],
    "VehPower": train["VehPower"].astype(str),
    "Exposure band": pd.cut(train["Exposure"], EXPOSURE_EDGES),
}
everyone = np.ones(len(train), dtype=bool)
for name, key in keys.items():
    print(f"=== all policies: {name} ===")
    print(report(everyone, key).to_string(), "\n")

age0 = (train["VehAge"] == 0).to_numpy()
print("age-0 vehicles: share of the remaining gap", round(diff_x[age0].sum() / diff_x.sum(), 3), "(was 0.62 before the interactions)")
for name in ["DrivAge band", "Brand", "VehGas", "BonusMalus class"]:
    print(f"=== inside age-0 vehicles: {name} ===")
    print(report(age0, keys[name]).to_string(), "\n")

remaining gap per policy: 0.00455
=== all policies: VehAge band ===
       policies  share_of_policies  share_of_gap  gap_vs_avg  actual/GLMx  actual/GBM
key                                                                                  
00        45992              0.085         0.480       5.647        1.001       1.007
01        57020              0.105         0.053       0.502        1.000       0.987
02        47155              0.087         0.020       0.229        1.001       1.016
03-05    105951              0.196         0.088       0.449        1.001       0.988
06-10    136831              0.253         0.174       0.688        0.999       1.002
11-15    107014              0.198         0.135       0.681        1.001       1.004
16+       41469              0.077         0.051       0.670        1.001       1.007 

=== all policies: DrivAge band ===
        policies  share_of_policies  share_of_gap  gap_vs_avg  actual/GLMx  actual/GBM
key                               

In [3]:
INTERACTIONS = {
    "age0_B12":         lambda d, X: ((d["VehAge"] == 0) & (d["VehBrand"] == "B12")).astype(int),
    "age0_Regular":     lambda d, X: ((d["VehAge"] == 0) & (d["VehGas"] == "Regular")).astype(int),
    "age0_off_ladder":  lambda d, X: ((d["VehAge"] == 0) & (X["bm_off_ladder"] == 1)).astype(int),
    "age0_bm100":       lambda d, X: ((d["VehAge"] == 0) & (X["bm_is_100"] == 1)).astype(int),
    # new in this round
    "age0_B12_Regular": lambda d, X: ((d["VehAge"] == 0) & (d["VehBrand"] == "B12") & (d["VehGas"] == "Regular")).astype(int),
    "age0_dage":        lambda d, X: (d["VehAge"] == 0) * (d["DrivAge"] - 38) / 10,                  # fixed centre: no learning
    "age0_dage2":       lambda d, X: (d["VehAge"] == 0) * ((d["DrivAge"] - 38) / 10) ** 2,          # inverted-U shape
    "offladder_slope":  lambda d, X: X["bm_off_ladder"] * (X["logBonusMalus"] - np.log(60)),         # uplift varies with the BM level
}
NUMERIC_EXTRAS = {"age0_dage", "age0_dage2", "offladder_slope"}      # continuous terms are scaled; 0/1 terms pass through

def features_x(d, extras):
    X = add_features(d)
    for name in extras:
        X[name] = INTERACTIONS[name](d, X)
    return X

def glm_x_fit_predict(tr, va, extras, alpha=1e-4):
    num_x = [e for e in extras if e in NUMERIC_EXTRAS]
    flag_x = [e for e in extras if e not in NUMERIC_EXTRAS]
    model = Pipeline([
        ("pre", ColumnTransformer([
            ("num",  StandardScaler(), NUM + num_x),
            ("flag", "passthrough", FLAGS + flag_x),
            ("cat",  OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=0.0037), CAT),
        ])),
        ("glm", PoissonRegressor(alpha=alpha, max_iter=1000)),
    ])
    model.fit(features_x(tr, extras), tr["ClaimNb"] / tr["Exposure"], glm__sample_weight=tr["Exposure"])
    return model.predict(features_x(va, extras)) * va["Exposure"].to_numpy()

V2 = ["age0_B12", "age0_Regular", "age0_off_ladder", "age0_bm100"]           # the current model ("GLM v2")
s_v2 = fold_scores(pd.read_csv(PROCESSED / "oof_glm_x.csv")["pred_claims"].to_numpy())
print(f"GLM {s_base.mean():.4f} | GLM v2 {s_v2.mean():.4f} | boosting {s_gbm.mean():.4f}")      # expect 0.3121 | 0.3083 | 0.3038

VARIANTS = {
    "v2 + B12 x petrol (3-way)": V2 + ["age0_B12_Regular"],
    "v2 + age0 x driver age":    V2 + ["age0_dage", "age0_dage2"],
    "v2 + off-ladder slope":     V2 + ["offladder_slope"],
    "v2 + all three":            V2 + ["age0_B12_Regular", "age0_dage", "age0_dage2", "offladder_slope"],
}
rows, oofs2 = [], {}
for name, extras in VARIANTS.items():
    oofs2[name], s = cross_validate(train, lambda tr, va, ex=extras: glm_x_fit_predict(tr, va, ex), check=False)
    rows.append({"variant": name, "mean_deviance": s.mean(), "gain_vs_v2": (s_v2 - s).mean(),
                 "folds_better": int((s < s_v2).sum()), "worst_fold_gain": (s_v2 - s).min(),
                 "total_gap_closed": (s_base - s).mean() / (s_base - s_gbm).mean()})
    print(name, "done")
print(pd.DataFrame(rows).round(4).to_string(index=False))

GLM 0.3121 | GLM v2 0.3083 | boosting 0.3038
v2 + B12 x petrol (3-way) done
v2 + age0 x driver age done
v2 + off-ladder slope done
v2 + all three done
                  variant  mean_deviance  gain_vs_v2  folds_better  worst_fold_gain  total_gap_closed
v2 + B12 x petrol (3-way)         0.3081      0.0003             5           0.0001            0.4844
   v2 + age0 x driver age         0.3082      0.0001             4          -0.0000            0.4644
    v2 + off-ladder slope         0.3083      0.0000             4          -0.0000            0.4528
           v2 + all three         0.3080      0.0004             5           0.0002            0.4967
